# 08 — Tableau Export

**Purpose:** Export analysis-ready datasets as Tableau Hyper extracts, verify each export, and provide a dashboard build guide.

**Inputs:**
- `data/features/record_inventory.csv`
- `data/features/train_features.csv`
- `data/features/test_features.csv`
- `data/features/apnea_episode_stats.csv`
- `models/xgb_best.joblib`

**Outputs:**
- `tableau/record_inventory.hyper`
- `tableau/train_features.hyper`
- `tableau/test_predictions.hyper`
- `tableau/episode_stats.hyper`
- `tableau/dashboard_spec.md`

![Status: COMPLETE](https://img.shields.io/badge/Status-COMPLETE-brightgreen)

In [ ]:
# ── Cell 1: Imports and paths ─────────────────────────────────────────────────
import pathlib, warnings
import numpy as np
import pandas as pd
import joblib
import pantab
from tableauhyperapi import TableName

warnings.filterwarnings('ignore')

FEATURES_DIR = pathlib.Path('../data/features')
MODELS_DIR   = pathlib.Path('../models')
TABLEAU_DIR  = pathlib.Path('../tableau')
TABLEAU_DIR.mkdir(parents=True, exist_ok=True)

def _export_hyper(df, hyper_path, schema='Extract', table='data'):
    """Write df to a Hyper file and read it back for verification."""
    tn = TableName(schema, table)
    # pantab requires string columns to use 'object' dtype —
    # convert numeric NaNs and ensure no mixed types
    df_out = df.copy()
    for col in df_out.select_dtypes(include='object').columns:
        df_out[col] = df_out[col].astype(str)
    pantab.frame_to_hyper(df_out, str(hyper_path), table=tn)
    df_back = pantab.frame_from_hyper(str(hyper_path), table=tn)
    size_kb = hyper_path.stat().st_size / 1024
    print(f'  {hyper_path.name:<35}  {len(df_back):>7,} rows  '
          f'{len(df_back.columns):>3} cols  {size_kb:>8.1f} KB')
    return df_back

print('Tableau export helper ready.')

In [ ]:
# ── Cell 2: Export record_inventory.hyper ────────────────────────────────────
def _severity(frac):
    if pd.isna(frac): return 'Unknown'
    if frac < 0.10:   return 'Mild'
    if frac < 0.30:   return 'Moderate'
    return 'Severe'

inventory = pd.read_csv(FEATURES_DIR / 'record_inventory.csv')
inventory['severity_category'] = inventory['apnea_fraction'].apply(_severity)

print('Exporting record_inventory.hyper ...')
path = TABLEAU_DIR / 'record_inventory.hyper'
_back = _export_hyper(inventory, path)
print(f'  Columns: {list(_back.columns)}')

In [ ]:
# ── Cell 3: Export train_features.hyper ──────────────────────────────────────
train_feat = pd.read_csv(FEATURES_DIR / 'train_features.csv')

print('Exporting train_features.hyper ...')
path = TABLEAU_DIR / 'train_features.hyper'
_back = _export_hyper(train_feat, path)

In [ ]:
# ── Cell 4: Export test_predictions.hyper ────────────────────────────────────
model     = joblib.load(MODELS_DIR / 'xgb_best.joblib')
ALL_FEATS = model.get_booster().feature_names
if ALL_FEATS is None:
    ALL_FEATS = ['mean_rr', 'sdnn', 'rmssd', 'pnn50',
                 'lf_power', 'hf_power', 'lf_hf_ratio', 'sd1', 'sd2',
                 'minutes_since_last_apnea', 'apnea_run_length',
                 'apnea_burden', 'prev_apnea']

test_feat = pd.read_csv(FEATURES_DIR / 'test_features.csv')
test_feat['prev_apnea'] = (test_feat['prev_label'] == 'A').astype(float)
test_clean = test_feat.dropna(subset=ALL_FEATS).copy()

test_clean['predicted_proba']  = model.predict_proba(test_clean[ALL_FEATS].values)[:, 1]
test_clean['predicted_label']  = model.predict(test_clean[ALL_FEATS].values)
test_clean['predicted_label']  = test_clean['predicted_label'].map({1: 'A', 0: 'N'})
test_clean['correct']          = (test_clean['predicted_label'] == test_clean['label'])

export_cols = ['record_id', 'minute_index', 'label', 'predicted_label',
               'predicted_proba', 'correct',
               'mean_rr', 'sdnn', 'rmssd', 'pnn50',
               'lf_power', 'hf_power', 'lf_hf_ratio', 'sd1', 'sd2']
export_cols = [c for c in export_cols if c in test_clean.columns]
test_export = test_clean[export_cols].rename(columns={'label': 'true_label'})

print('Exporting test_predictions.hyper ...')
path = TABLEAU_DIR / 'test_predictions.hyper'
_back = _export_hyper(test_export, path)

In [ ]:
# ── Cell 5: Export episode_stats.hyper ───────────────────────────────────────
episode_stats = pd.read_csv(FEATURES_DIR / 'apnea_episode_stats.csv')

print('Exporting episode_stats.hyper ...')
path = TABLEAU_DIR / 'episode_stats.hyper'
_back = _export_hyper(episode_stats, path)

In [ ]:
# ── Cell 6: Export manifest with file sizes ───────────────────────────────────
print('\nTABLEAU EXPORT MANIFEST')
print('=' * 62)
print(f'{"File":<35}  {"Size (KB)":>10}  {"Status"}')
print('-' * 62)

hyper_files = sorted(TABLEAU_DIR.glob('*.hyper'))
for hp in hyper_files:
    size_kb = hp.stat().st_size / 1024
    print(f'{hp.name:<35}  {size_kb:>10.1f}  OK')

if not hyper_files:
    print('  No .hyper files found — re-run export cells above.')
print('=' * 62)
print(f'Total exports: {len(hyper_files)}')

## Tableau Dashboard Build Guide

This section documents how to recreate the **Sleep Apnea Detection — Model Evaluation Dashboard** in Tableau Desktop using the four exported Hyper files.

---

### Data Sources
Connect to each file via **Data → New Data Source → Tableau Server / More → Hyper**:

| File | Sheets that use it |
|------|--------------------|
| `tableau/record_inventory.hyper` | Sheet 1 |
| `tableau/train_features.hyper` | Sheet 2 |
| `tableau/test_predictions.hyper` | Sheets 3, 4 |
| `tableau/episode_stats.hyper` | (optional supplement) |

---

### Sheet 1 — Subject Overview  
**Data source:** `record_inventory.hyper`

1. Drag `record_id` to **Columns**, `apnea_burden` (×100) to **Rows** → bar chart.
2. Drag `severity_category` to **Color**. Set palette: `Mild`=#27ae60, `Moderate`=#f1c40f, `Severe`=#e74c3c, `Unknown`=#bdc3c7.
3. Sort bars descending by `apnea_burden`.
4. Add **Tooltip** fields: `record_id`, `group`, `n_apnea_minutes`, `severity_category`.
5. Set title: **"Apnea Burden by Subject"**.

---

### Sheet 2 — HRV Feature Explorer  
**Data source:** `train_features.hyper`

1. Drag `lf_hf_ratio` to **Columns**, `rmssd` to **Rows** → scatter.
2. Drag `label` to **Color**: `A` (Apnea) = #F44336, `N` (Normal) = #2196F3. Set **Opacity** = 50%.
3. Add `record_id` to **Filters** → show as a dropdown filter control.
4. Set title: **"HRV Feature Space by Apnea Label"**.

---

### Sheet 3 — Model Performance by Subject  
**Data source:** `test_predictions.hyper`

1. Create a calculated field:
   `Minute Accuracy = SUM([correct]) / COUNT([correct])`
2. Drag `record_id` to **Rows**, `Minute Accuracy` to **Columns** → horizontal bar chart.
3. Sort descending by `Minute Accuracy`.
4. Color rule (create bins or use conditional calculated field):
   - `IF [Minute Accuracy] > 0.85 THEN 'High' ELSEIF [Minute Accuracy] > 0.70 THEN 'Medium' ELSE 'Low' END`
   - Map: `High`=#27ae60, `Medium`=#f1c40f, `Low`=#e74c3c.
5. Add a **Reference Line** at the overall test accuracy (enter as a constant).
6. Set title: **"Per-Subject Prediction Accuracy"**.

---

### Sheet 4 — Apnea Probability Timeline  
**Data source:** `test_predictions.hyper`

1. Drag `minute_index` to **Columns**, `predicted_proba` to **Rows** → line chart.
2. Add a **Band** / area annotation: create calculated field `Apnea Band = IF [true_label]='A' THEN 1 ELSE NULL END` and add to a dual axis, set to area mark type with low opacity red.
3. Add `record_id` to **Filters** → dropdown.
4. Set title: **"Predicted Apnea Probability vs Ground Truth"**.

---

### Dashboard Layout

1. Create a new dashboard, set size to **1600 × 900**.
2. Arrange sheets in a **2×2 grid**:
   - Top-left: Sheet 1 (Subject Overview)
   - Top-right: Sheet 2 (HRV Feature Explorer)
   - Bottom-left: Sheet 3 (Model Performance by Subject)
   - Bottom-right: Sheet 4 (Probability Timeline)
3. Add a **Title banner** (text object at top, full width):
   `Sleep Apnea Detection — Model Evaluation Dashboard`
4. Add a **Text box** (bottom or side):
   ```
   Dataset: PhysioNet Apnea-ECG (Penzel et al.)
   70 overnight ECG recordings, 100 Hz, per-minute apnea annotations.
   Model: XGBoost classifier on 13 HRV + episode-context features.
   ```
5. Enable **filter actions** between Sheet 3 and Sheet 4 (clicking a subject row in Sheet 3 filters the timeline in Sheet 4).

---

*See `tableau/dashboard_spec.md` for the machine-readable version of this specification.*